# Evaluation Hors-Domaine (Domain Shift)
## Generalisation des modeles entraines sur LIAR

**Objectif** : Tester la robustesse des modeles entraines sur LIAR sur **deux datasets externes** :

| Dataset | Domaine | Taille | Caracteristique |
|---|---|---|---|
| **ISOT** | News politiques generales | ~34k | Articles complets, Reuters + sites fake |
| **WELFake** | News diverses (mix de sources) | ~64k | Plus large variete thematique |

Cette evaluation revele comment les modeles **se generalisent** au-dela du domaine d'entrainement (LIAR = courtes declarations politiques).

In [3]:
import pandas as pd
import numpy as np
import json
import joblib
import re
from pathlib import Path

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from xgboost import XGBClassifier
import torch

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

Device : cpu


## 1. Chargement des deux datasets externes

Les datasets sont deja pretraites (`data/Traitees/`).
On harmonise les colonnes en `text` et `label_binary` (0=Fake, 1=Real).

In [4]:
SAMPLE_SIZE = 5000

# --- 1. ISOT (articles complets) ---
# Lire colonnes essentielles uniquement, en string numpy (evite pyarrow memory)
df_isot = pd.read_parquet(
    DATA_DIR / "isot_clean.parquet",
    columns=["text_clean", "label"],
    dtype_backend="numpy_nullable",
)
df_isot = df_isot.rename(columns={"text_clean": "text", "label": "label_binary"})
df_isot = df_isot.dropna(subset=["text"])
# Sample AVANT filtrage longueur (evite blowup memoire)
if len(df_isot) > SAMPLE_SIZE:
    df_isot = df_isot.sample(SAMPLE_SIZE, random_state=42)
df_isot["text"] = df_isot["text"].astype(str)
df_isot = df_isot[df_isot["text"].str.len() > 10].reset_index(drop=True)

# --- 2. WELFake (mix de sources) ---
df_wel = pd.read_parquet(
    DATA_DIR / "welfake_clean.parquet",
    columns=["text_clean", "label"],
    dtype_backend="numpy_nullable",
)
df_wel = df_wel.rename(columns={"text_clean": "text", "label": "label_binary"})
df_wel = df_wel.dropna(subset=["text"])
# Sample AVANT filtrage longueur (evite ArrowMemoryError)
if len(df_wel) > SAMPLE_SIZE:
    df_wel = df_wel.sample(SAMPLE_SIZE, random_state=42)
df_wel["text"] = df_wel["text"].astype(str)
df_wel = df_wel[df_wel["text"].str.len() > 10].reset_index(drop=True)

DATASETS = {
    "ISOT": df_isot,
    "WELFake": df_wel,
}

for name, df in DATASETS.items():
    n_fake = (df["label_binary"] == 0).sum()
    n_real = (df["label_binary"] == 1).sum()
    avg_len = df["text"].str.len().mean()
    print(f"{name:10s} : {len(df):5d} echantillons (Fake: {n_fake}, Real: {n_real}) -- longueur moyenne: {avg_len:.0f} chars")


ISOT       :  5000 echantillons (Fake: 2526, Real: 2474) -- longueur moyenne: 2452 chars
WELFake    :  4944 echantillons (Fake: 2742, Real: 2202) -- longueur moyenne: 3342 chars


## 2. Chargement des modeles entraines sur LIAR

On charge :
1. **Baseline TF-IDF + LogReg** -- pour reference
2. **DistilBERT fine-tune** (tete de classification HuggingFace)
3. **DistilBERT + XGBoost** (notre modele final) -- backbone DistilBERT + XGBoost sur embeddings + metadata

In [5]:
# 1. Baseline TF-IDF + LogReg
baseline_model = joblib.load(MODELS_DIR / "tfidf_logreg_binary.joblib")
print("Baseline TF-IDF + LogReg charge")

# 2. DistilBERT fine-tune
distilbert_path = MODELS_DIR / "distilbert_best"
if distilbert_path.exists():
    tokenizer_db = AutoTokenizer.from_pretrained(str(distilbert_path))
    model_db = AutoModelForSequenceClassification.from_pretrained(str(distilbert_path))
    model_db.to(device); model_db.eval()
    print("DistilBERT fine-tune charge")
    HAS_DISTILBERT = True
else:
    print("DistilBERT non disponible")
    HAS_DISTILBERT = False

# 3. DistilBERT + XGBoost (modele final)
xgb_path = MODELS_DIR / "distilbert_xgboost.joblib"
bundle_path = MODELS_DIR / "distilbert_xgboost_meta.json"
if xgb_path.exists() and bundle_path.exists() and HAS_DISTILBERT:
    xgb_model = joblib.load(xgb_path)
    with open(bundle_path) as f:
        meta_bundle = json.load(f)
    print(f"DistilBERT + XGBoost charge (test_acc={meta_bundle['test_acc']:.4f}, F1={meta_bundle['test_f1_weighted']:.4f})")
    HAS_XGB = True
else:
    print("DistilBERT + XGBoost non disponible -- relancez Modeles_Avances.ipynb")
    HAS_XGB = False

Baseline TF-IDF + LogReg charge
DistilBERT fine-tune charge
DistilBERT + XGBoost charge (test_acc=0.6654, F1=0.6648)


## 3. Evaluation cross-dataset

On applique chaque modele sur les trois datasets et on stocke les metriques.

In [6]:
def predict_transformer(texts, tokenizer, model, device, batch_size=32):
    """Predictions par batch avec un transformer (tete de classification)."""
    all_preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                               padding=True, max_length=256).to(device)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def extract_mean_embeddings(texts, tokenizer, model, device, batch_size=32):
    """Mean pooling embeddings depuis le backbone DistilBERT."""
    model.eval()
    backbone = model.distilbert if hasattr(model, "distilbert") else model.base_model
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors="pt", truncation=True,
                            padding=True, max_length=256).to(device)
            out = backbone(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            mean_pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(mean_pooled.cpu().numpy())
    return np.vstack(embeddings)


def build_default_meta(texts, bundle):
    """Construit les features metadata avec valeurs par defaut (speaker/party inconnus en OOD)."""
    n = len(texts)
    gm = bundle["global_mean"]
    return np.column_stack([
        np.full(n, 0.5),                                # credibility_score (inconnu -> 0.5)
        np.full(n, gm),                                  # speaker_cred -> moyenne globale
        np.zeros(n),                                     # log(speaker_count+1) = 0
        np.full(n, gm),                                  # party_cred -> moyenne globale
        np.array([len(t.split()) for t in texts]),       # n_words
    ])


# Stockage : ood_results[dataset][model] = {accuracy, f1}
ood_results = {}

for ds_name, df in DATASETS.items():
    print(f"\n========== {ds_name} ==========")
    texts = df["text"].tolist()
    labels = df["label_binary"].values

    ood_results[ds_name] = {}

    # --- Baseline TF-IDF + LogReg ---
    y_pred = baseline_model.predict(texts)
    acc = accuracy_score(labels, y_pred)
    f1 = f1_score(labels, y_pred, average="weighted")
    print(f"TF-IDF + LogReg     -- Acc: {acc:.4f}  F1: {f1:.4f}")
    ood_results[ds_name]["TF-IDF + LogReg"] = {"accuracy": round(acc, 4), "f1": round(f1, 4)}

    # --- DistilBERT fine-tune ---
    if HAS_DISTILBERT:
        y_pred = predict_transformer(texts, tokenizer_db, model_db, device)
        acc = accuracy_score(labels, y_pred)
        f1 = f1_score(labels, y_pred, average="weighted")
        print(f"DistilBERT          -- Acc: {acc:.4f}  F1: {f1:.4f}")
        ood_results[ds_name]["DistilBERT"] = {"accuracy": round(acc, 4), "f1": round(f1, 4)}

    # --- DistilBERT + XGBoost (modele final) ---
    if HAS_XGB:
        emb = extract_mean_embeddings(texts, tokenizer_db, model_db, device)
        meta = build_default_meta(texts, meta_bundle)
        X_combined = np.hstack([emb, meta])
        y_pred = xgb_model.predict(X_combined)
        acc = accuracy_score(labels, y_pred)
        f1 = f1_score(labels, y_pred, average="weighted")
        print(f"DistilBERT + XGBoost -- Acc: {acc:.4f}  F1: {f1:.4f}")
        ood_results[ds_name]["DistilBERT + XGBoost"] = {"accuracy": round(acc, 4), "f1": round(f1, 4)}

print("\n=== Evaluation cross-dataset terminee ===")


========== ISOT ==========
TF-IDF + LogReg     -- Acc: 0.5374  F1: 0.5368
DistilBERT          -- Acc: 0.7362  F1: 0.7358
DistilBERT + XGBoost -- Acc: 0.6666  F1: 0.6538

========== WELFake ==========
TF-IDF + LogReg     -- Acc: 0.4343  F1: 0.4329
DistilBERT          -- Acc: 0.2399  F1: 0.2385
DistilBERT + XGBoost -- Acc: 0.3556  F1: 0.3344

=== Evaluation cross-dataset terminee ===


## 4. Comparaison In-Domain (LIAR) vs Out-of-Domain

In [7]:
# Charger metriques in-domain (LIAR test set)
metrics_path = MODELS_DIR / "baselines_metrics.json"
in_domain = {}
if metrics_path.exists():
    with open(metrics_path) as f:
        baseline_metrics = json.load(f)
    if "TF-IDF + LogReg" in baseline_metrics:
        in_domain["TF-IDF + LogReg"] = {
            "accuracy": baseline_metrics["TF-IDF + LogReg"].get("test_acc", 0),
            "f1": baseline_metrics["TF-IDF + LogReg"].get("test_f1", 0),
        }

# Construire tableau : une ligne par (modele, dataset)
rows = []
first_ds = list(DATASETS.keys())[0]
for model_name in ood_results[first_ds]:
    for ds_name in DATASETS:
        rows.append({
            "Modele": model_name,
            "Dataset": ds_name,
            "Accuracy": ood_results[ds_name][model_name]["accuracy"],
            "F1": ood_results[ds_name][model_name]["f1"],
        })

ood_df = pd.DataFrame(rows)
print("=== Performance Out-of-Domain par dataset ===")
print(ood_df.to_string(index=False))

# Tableau pivot Accuracy
print("\n=== Pivot Accuracy ===")
pivot_acc = ood_df.pivot(index="Modele", columns="Dataset", values="Accuracy")
if "TF-IDF + LogReg" in in_domain:
    pivot_acc.insert(0, "LIAR (in-domain)", [in_domain["TF-IDF + LogReg"]["accuracy"] if m == "TF-IDF + LogReg" else None for m in pivot_acc.index])
print(pivot_acc)

=== Performance Out-of-Domain par dataset ===
              Modele Dataset  Accuracy     F1
     TF-IDF + LogReg    ISOT    0.5374 0.5368
     TF-IDF + LogReg WELFake    0.4343 0.4329
          DistilBERT    ISOT    0.7362 0.7358
          DistilBERT WELFake    0.2399 0.2385
DistilBERT + XGBoost    ISOT    0.6666 0.6538
DistilBERT + XGBoost WELFake    0.3556 0.3344

=== Pivot Accuracy ===
Dataset               LIAR (in-domain)    ISOT  WELFake
Modele                                                 
DistilBERT                         NaN  0.7362   0.2399
DistilBERT + XGBoost               NaN  0.6666   0.3556
TF-IDF + LogReg                 0.6006  0.5374   0.4343


In [8]:
# Visualisation : performance par dataset
fig = go.Figure()

datasets_viz = list(DATASETS.keys())
first_ds = datasets_viz[0]
models_viz = list(ood_results[first_ds].keys())

colors = {
    "TF-IDF + LogReg": "#3498db",
    "DistilBERT": "#e67e22",
    "DistilBERT + XGBoost": "#e74c3c",
}

for model_name in models_viz:
    accs = [ood_results[ds][model_name]["accuracy"] for ds in datasets_viz]
    fig.add_trace(go.Bar(
        name=model_name,
        x=datasets_viz, y=accs,
        text=[f"{a:.2f}" for a in accs], textposition="outside",
        marker_color=colors.get(model_name, "#95a5a6"),
    ))

if "TF-IDF + LogReg" in in_domain:
    fig.add_hline(
        y=in_domain["TF-IDF + LogReg"]["accuracy"],
        line_dash="dash", line_color="green",
        annotation_text=f"LIAR baseline in-domain: {in_domain['TF-IDF + LogReg']['accuracy']:.3f}",
    )

fig.update_layout(
    title="Accuracy Out-of-Domain par dataset (modeles entraines sur LIAR)",
    yaxis_title="Accuracy",
    barmode="group",
    height=500,
)
fig.show()

## 5. Analyse des erreurs cross-domain

On observe la matrice de confusion par dataset pour comprendre **comment** le modele se trompe.

In [9]:
# Matrice de confusion par dataset (modele final : DistilBERT + XGBoost)
n_ds = len(DATASETS)
fig = make_subplots(
    rows=1, cols=n_ds,
    subplot_titles=list(DATASETS.keys()),
    horizontal_spacing=0.18,
)

for col, (ds_name, df) in enumerate(DATASETS.items(), start=1):
    y_true = df["label_binary"].values
    if HAS_XGB:
        emb = extract_mean_embeddings(df["text"].tolist(), tokenizer_db, model_db, device)
        meta = build_default_meta(df["text"].tolist(), meta_bundle)
        y_pred = xgb_model.predict(np.hstack([emb, meta]))
    else:
        y_pred = baseline_model.predict(df["text"].tolist())
    cm = confusion_matrix(y_true, y_pred)

    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=["Pred Fake", "Pred Real"],
            y=["True Fake", "True Real"],
            text=cm, texttemplate="%{text}",
            colorscale="Blues", showscale=False,
        ),
        row=1, col=col,
    )

fig.update_layout(
    title="Matrices de confusion -- DistilBERT + XGBoost (modele final) sur les datasets OOD",
    height=400,
)
fig.show()

In [10]:
# Comparaison de la longueur des textes : LIAR vs OOD
df_liar = pd.read_parquet(DATA_DIR / "liar_complet.parquet")
liar_lens = df_liar["statement"].str.len()

fig = go.Figure()
fig.add_trace(go.Box(y=liar_lens, name="LIAR (train)", marker_color="green"))
for ds_name, df in DATASETS.items():
    fig.add_trace(go.Box(y=df["text"].str.len(), name=ds_name))

fig.update_layout(
    title="Distribution de la longueur des textes (caracteres)",
    yaxis_title="Longueur (chars)",
    yaxis_type="log",
    height=500,
)
fig.show()

print("\nLongueur mediane par dataset :")
print(f"  LIAR       : {liar_lens.median():.0f} chars")
for ds_name, df in DATASETS.items():
    print(f"  {ds_name:11s}: {df['text'].str.len().median():.0f} chars")


Longueur mediane par dataset :
  LIAR       : 99 chars
  ISOT       : 2220 chars
  WELFake    : 2454 chars


## 6. Discussion et synthese du domain shift

**Modele final retenu** : **DistilBERT + XGBoost** -- combine la richesse semantique des embeddings DistilBERT et la puissance d'XGBoost pour la classification.

**Limites de l'evaluation OOD** :
- Les datasets externes (ISOT, WELFake) **ne contiennent pas** les colonnes `speaker`, `party` ni `credibility_score`.
- Les features metadata sont donc remplacees par des **valeurs par defaut** (moyenne globale de LIAR).
- Cela degrade les performances OOD du modele hybride, qui s'appuie partiellement sur ces signaux.

**Causes du domain shift** :
1. **Longueur des textes** : LIAR (~100 chars) vs ISOT/WELFake (>1000 chars).
2. **Vocabulaire** : LIAR-centre (politiciens US, sujets politiques) -- ne se transpose pas.
3. **Style** : declarations orales vs articles ecrits.
4. **Metadata absentes** : speaker/party/credibility inconnus en OOD.

**Conclusions** :
- Un modele performant in-domain peut chuter brutalement out-of-domain.
- DistilBERT generalise mieux que TF-IDF grace a ses representations contextuelles pre-entrainees.
- Le hybride DistilBERT + XGBoost reste competitif meme sans les metadata, mais une **adaptation de domaine** serait necessaire pour un deploiement multi-source.